# Hydrides Example

This notebook provides a study case of **crystALL** in aiding the scientific search for crystal structures. If you haven't completed it yet, please start at [`01.QuickStart.ipynb`](01.QuickStart.ipynb).


In this tutorial, we investigate the ocurrence of boron hydrides, as H-rich compounds for hydrogen storage applications. In addition, we select `ABX3` compositions, aiming to quickly assess whether these compounds can exhibit perovskite-type structures, which are targeted due to their interesting properties for H2 storage, such as high kinetics and tolerance to defects.

Herein, we initiate our `CrystalSearch` object with the NaMg(BH4)3 composition:

In [ ]:
%%capture
from crystall import CrystALLSearch

In [ ]:
HydrideSearch = CrystALLSearch("NaMg(BH4)3")

We then connect to MPDD and search for possible prototypes:

In [ ]:
from pymongo import MongoClient
import os; client_string = os.getenv('MONGODB_URI')

MPDD_database = MongoClient(client_string)['MPDD']
HydrideSearch.connect_MPDD(MPDD_database)

HydrideSearch.generate_MPDD(limit=2000)

And calculate the pySIPFENN predictors for clusterization and classification:

In [ ]:
HydrideSearch.run_pysipfenn()

We use `filter_prototypes` to down select the initial pool of prototypes generated by `CrystALLSearch` and calculate their ConvexHull stability using `get_mpdd_convexhull`.

In [ ]:
HydrideSearch.filter_prototypes(max_prototypes=100)
HydrideSearch.get_mpdd_convexhull()

To categorize each protoypes according to its physical features, we run `cluster_prototypes_KMeans` using `n_cluster=7`. 

Note that we don't need to assign any number of clusters in order to benefit from the unsupervised cluster assignment of **crystALL**. The automatic assignment of n clusters in `cluster_prototypes_KMeans` provides an educated initial guess about the categorization of our prototype set. 

In [ ]:
HydrideSearch.cluster_prototypes_KMeans(n_clusters=7)

Finally, we can export and visualize the data after clusterization using `plot_2D`, `plot_3D`, and `export_table`:

In [ ]:
HydrideSearch.plot_2D()
HydrideSearch.plot_3D()
HydrideSearch.export_table(content = ['all-prototypes', 'top-candidates'])

In this example, the 7 clusters identified group together prototypes with similar atomic environments, representing primarily hexagonal structures (clusters 1 and 2), triclinic structures (cluster 3), trigonal structures (clusters 4, 5, and 7), and perovskite-type structures (cluster 6).

Note that closer inspection of each cluster's structures may reveal additional subdomains. In such cases, users are encouraged to refine their results by adjusting n clusters in `cluster_prototypes_KMeans`. 

The code below renders the 3D visualizations of each cluster's top candidate identified.

In [ ]:
import pyvista as pv
from ipywidgets import HBox, VBox, HTML
from IPython.display import display

pv.start_xvfb()

# Function to create a plotter and import a GLB file
def create_plotter(file_path):
    pl = pv.Plotter(notebook=True, lighting='three lights', window_size=[400, 400])
    
    mesh = pv.read(file_path)
    pl.add_mesh(mesh, style='surface', show_edges=True, edge_color='black', line_width=0.1)
    pl.import_gltf(file_path)
    pl.camera.zoom(1.5)
    return pl

# Create three plotters with different GLB files
plotter1 = create_plotter("3Dmodels/CaTh(SeO4)3-hexagonal-gnome723a502352-p1.glb")
plotter2 = create_plotter("3Dmodels/RbTa(SO4)3-triclinic-gnomed5318fab9f-p2.glb")
plotter3 = create_plotter("3Dmodels/HoEr(MoO4)3-rhombohedral-gnomea32845676f-p1.glb")
plotter4 = create_plotter("3Dmodels/TiRe(PO4)3-trigonal-mp1043756-p1.glb")
plotter5 = create_plotter("3Dmodels/NbV(PO4)3-triclinic-gnomea9b189292f-p2.glb")
plotter6 = create_plotter("3Dmodels/Cr3InFeO12-trigonal-gnome0333a83d4d-p1.glb")

# Create a VBox for each plotter that includes the title and the 3D viewer
vbox1 = VBox([HTML("<center>CaTh(SeO4)3 / ThCr3HgO12</center>"), plotter1.show(jupyter_backend='html', return_viewer=True)])
vbox2 = VBox([HTML("<center>RbTa(SO4)3</center>"), plotter2.show(jupyter_backend='html', return_viewer=True)])
vbox3 = VBox([HTML("<center>HoEr(MoO4)3</center>"), plotter3.show(jupyter_backend='html', return_viewer=True)])
vbox4 = VBox([HTML("<center>TiRe(PO4)3</center>"), plotter4.show(jupyter_backend='html', return_viewer=True)])
vbox5 = VBox([HTML("<center>NbV(PO4)3</center>"), plotter5.show(jupyter_backend='html', return_viewer=True)])
vbox6 = VBox([HTML("<center>Cr3InFeO12</center>"), plotter6.show(jupyter_backend='html', return_viewer=True)])

# Display the combined widget
display(HBox([vbox1, vbox3, vbox5]))
display(HBox([vbox2, vbox4, vbox6]))

Notably, the top candidate structure predicted in cluster 6, based on the NbV(PO4)3 entry from the Gnome dataset (a9b189292f), comprises a highly distorted perovskite structure, where the A sites are coordinated by 12 BH4⁻ anions, with half pointing inward and half outward. The corner-sharing coordination exhibited by this prototype represents an unprecedented borohydride structure, distinct from previously reported structures, which are characterized by edge-sharing ([Schouwink2015](https://doi.org/10.3390/en8088286)) and face-sharing ([Moller2016](https://doi.org/10.1039/c5dt03590b)) configurations.

The NaMg(BH₄)₃ case highlights the significant potential of **crystALL** not only in quickly and effortlessly predicting candidate structures that would otherwise remain unknown, but also in providing valuable insights into their relative stability compared to other possible structures. Once identified, the predicted structures can be locally relaxed using first-principles calculations, allowing for further modeling of thermal stability and computational characterization to aid in experimental validation.